# Verification of mod 25 classification of eigenforms
For the faster option of using the precomputed source data, set USE_ARCHIVED_SOURCE_DATA = True, Otherwise, set it to FALSE to recompute the source data.

In [2]:
import sys
from pathlib import Path

PYTHON_DIRECTORY = Path("python").resolve()
if str(PYTHON_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIRECTORY))

from hecke_congruences import *
from load_source_data import load_source_data

USE_ARCHIVED_SOURCE_DATA = True
SOURCE_DATA_DIRECTORY = Path("source_data")
MOD25_SOURCE_ARCHIVE = (
    SOURCE_DATA_DIRECTORY / "p5_mod25_T2_T19_all_degrees.npz"
)

def get_source_data(R, d, q, archive):
    """Load the archived source when requested; otherwise construct it over R.

    Keep the stated source scope, cyclic orders and orientation.
    """
    if USE_ARCHIVED_SOURCE_DATA:
        return load_source_data(R, d, q, archive)

    return prepare_source_data(R, d, q)

In [3]:
p = 5
m = 2
modulus = p^m
R = Integers(modulus)

P.<X> = PolynomialRing(R)
J.<U,V> = PolynomialRing(R, 2)

period = euler_phi(p^m)
a_m = p^m * (p - 1)
b_m = p^(m - 1) * (p + 1)
surjectivity_bound = a_m + b_m

exact_induction_base = tuple(
    range(b_m, surjectivity_bound, 2)
)

lower_base_degrees = tuple(
    range(0, b_m, 2)
)

degree_residues = tuple(range(0, period, 2))

beta = {
    0: 4,
    2: 0,
    4: 2,
    6: 0,
    8: 4,
    10: 1,
    12: 0,
    14: 3,
    16: 0,
    18: 1,
}

c = {
    r: 1 if r % 4 == 0 else 2
    for r in degree_residues
}

delta_gamma = {
    0:  {0: (1, 4), 1: (4, 0)},
    2:  {0: (4, 4), 1: (1, 3)},
    4:  {0: (1, 1), 1: (4, 2)},
    6:  {0: (4, 2), 1: (1, 1)},
    8:  {0: (1, 3), 1: (4, 4)},
    10: {0: (4, 0), 1: (1, 4)},
    12: {0: (1, 0), 1: (4, 1)},
    14: {0: (4, 3), 1: (1, 2)},
    16: {0: (1, 2), 1: (4, 3)},
    18: {0: (4, 1), 1: (1, 0)},
}

def affine_factor(r, eta):
    """Form the specified affine polynomial factor in the chosen coefficient ring."""
    delta, gamma = delta_gamma[r][eta]
    return (U - delta * (c[r] * V + 3 - eta) - 5 * gamma)

G = {
    r: affine_factor(r, 0) * affine_factor(r, 1)
    for r in degree_residues
}

Q19 = {
    r: X - 5 * beta[r]
    for r in degree_residues
}

F19 = (X^3 + X)^2

print("Dickson degrees:", a_m, b_m)
print("degree residues:", degree_residues)
print("lower verification range:", lower_base_degrees)
print("exact induction base:", exact_induction_base)

Dickson degrees: 100 30
degree residues: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18)
lower verification range: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28)
exact induction base: (30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126, 128)


In [4]:
def verify_mod25_case(d, q):
    """Check the displayed ordinary joint and divided selector relations in this degree and orientation.

    Prepare the source once and reuse its coordinates and Hecke actions.
    """
    data = get_source_data(R, d, q, MOD25_SOURCE_ARCHIVE)

    relation_residue = (
        d + 10 * (q % 2)
    ) % period

    joint = verify_ordinary_joint_identities(
        F=G[relation_residue],
        hecke_indices=(19, 2),
        data=data,
        check_descent=False,
    )

    divided = verify_divided_identities(
        F=F19,
        Q=Q19[relation_residue],
        n=19,
        a=1,
        b=1,
        data=data,
        check_descent=False,
    )

    return {
        "degree": d,
        "orientation": q,
        "sign": data["sign"],
        "relation_residue": relation_residue,
        "joint": joint,
        "divided": divided,
        "passed": (
            joint["passed"]
            and divided["passed"]
        ),
    }

In [6]:
cases = (
    [
        (d, q)
        for d in sorted(exact_induction_base, reverse=True)
        for q in range(0, p - 1)
    ]
    + [
        (d, 0)
        for d in sorted(lower_base_degrees, reverse=True)
    ]
)
results = []

for d, q in cases:
    test = verify_mod25_case(d, q)
    results.append(test)

    joint = test["joint"]
    divided = test["divided"]

    print(
        f"d={d:3d}, "
        f"r={test['relation_residue']:2d}, "
        f"q={q}, "
        f"sign={test['sign']:+d}, "
        f"rank={joint['rank']:3d}, "
        f"joint={joint['passed']}, "
        f"division={divided['division_passed']}, "
        f"terminal={divided['terminal_passed']}, "
        f"route={divided['verification_route']}, "
        f"passed={test['passed']}"
    )

    assert test["passed"]

print("——————————————————————————————————————————————")
print("cases completed:", len(results))
print("ALL MODULO-25 IDENTITIES VERIFIED")

d=128, r= 8, q=0, sign=+1, rank= 11, joint=True, division=True, terminal=True, route=global_scaled_annihilation, passed=True
d=128, r=18, q=1, sign=-1, rank= 12, joint=True, division=True, terminal=True, route=global_scaled_annihilation, passed=True
d=128, r= 8, q=2, sign=+1, rank= 11, joint=True, division=True, terminal=True, route=global_scaled_annihilation, passed=True
d=128, r=18, q=3, sign=-1, rank= 12, joint=True, division=True, terminal=True, route=global_scaled_annihilation, passed=True
d=126, r= 6, q=0, sign=+1, rank= 14, joint=True, division=True, terminal=True, route=global_scaled_annihilation, passed=True
d=126, r=16, q=1, sign=-1, rank= 10, joint=True, division=True, terminal=True, route=global_scaled_annihilation, passed=True
d=126, r= 6, q=2, sign=+1, rank= 14, joint=True, division=True, terminal=True, route=global_scaled_annihilation, passed=True
d=126, r=16, q=3, sign=-1, rank= 10, joint=True, division=True, terminal=True, route=global_scaled_annihilation, passed=True


# Check that each possible signature given by the identities above is realized by a strong eigenform

In [7]:
possible_signatures = {}

for k_residue in degree_residues:
    degree_residue = (k_residue - 2) % period
    possible_signatures[k_residue] = set()

    for a2 in range(modulus):
        for a19 in range(modulus):
            if G[degree_residue](R(a19), R(a2)) != 0:
                continue

            if (a19 - 5*beta[degree_residue]) % 5 != 0:
                continue

            z19 = ((a19 - 5*beta[degree_residue]) // 5) % 5

            if (z19^3 + z19) % 5 != 0:
                continue

            possible_signatures[k_residue].add((a2, a19))

print(
    f"{'k mod ' + str(period):<10} | "
    f"(a_2, a_19) mod {p^m}"
)
print("-" * 70)

for r in sorted(possible_signatures):
    pairs = ", ".join(
        f"({a2}, {a19})"
        for a2, a19 in sorted(possible_signatures[r])
    )

    print(f"{str(r):<10} | {pairs}")

k mod 20   | (a_2, a_19) mod 25
----------------------------------------------------------------------
0          | (6, 15), (9, 20), (11, 5), (14, 5), (16, 20), (19, 15)
2          | (3, 20), (7, 5), (12, 10), (13, 10), (18, 5), (22, 20)
4          | (1, 15), (4, 0), (9, 10), (16, 10), (21, 0), (24, 15)
6          | (2, 10), (8, 0), (12, 20), (13, 20), (17, 0), (23, 10)
8          | (4, 15), (9, 0), (11, 10), (14, 10), (16, 0), (21, 15)
10         | (2, 20), (8, 10), (12, 5), (13, 5), (17, 10), (23, 20)
12         | (1, 20), (4, 5), (9, 15), (16, 15), (21, 5), (24, 20)
14         | (3, 0), (7, 10), (12, 15), (13, 15), (18, 10), (22, 0)
16         | (6, 0), (9, 5), (11, 15), (14, 15), (16, 5), (19, 0)
18         | (2, 15), (3, 10), (12, 0), (13, 0), (22, 10), (23, 15)


In [8]:
def krw_reduction_to_integer(x, nf, pr, p, m):
    """Find an integer representative of the selected coefficient modulo the KRW ideal.

    Test equality in the prime-ideal quotient rather than assuming the residue is rational.
    """
    e = pr[2]
    N = e*(m - 1) + 1

    candidates = [
        r
        for r in range(p^m)
        if nf.idealval(x - r, pr) >= N
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"expected one residue in Z/{p^m}Z, "
            f"but found {candidates}"
        )

    return Integers(p^m)(candidates[0])

In [9]:
B = 142
l1, l2 = 2, 19

R = Integers(p^m)
signatures = []

for k in range(12, B + 1, 2):
    S = CuspForms(1, k)

    if S.dimension() == 0:
        continue

    print(f"calculating strong signatures at weight {k}")

    for j, f in enumerate(S.newforms(names='a')):
        K = f.base_ring()

        if K == QQ:
            signature = (k, R(f[l1]), R(f[l2]))
            signatures.append(signature)
            continue

        pol = K.pari_polynomial('y')
        nf = pari([pol, [p]]).nfinit(4)

        a1 = pari(f[l1])
        a2 = pari(f[l2])

        for pr in nf.idealprimedec(p):
            a1_bar = krw_reduction_to_integer(
                a1, nf, pr, p, m
            )

            a2_bar = krw_reduction_to_integer(
                a2, nf, pr, p, m
            )

            signatures.append((k, a1_bar, a2_bar))

calculating strong signatures at weight 12
calculating strong signatures at weight 16
calculating strong signatures at weight 18
calculating strong signatures at weight 20
calculating strong signatures at weight 22
calculating strong signatures at weight 24
calculating strong signatures at weight 26
calculating strong signatures at weight 28
calculating strong signatures at weight 30
calculating strong signatures at weight 32
calculating strong signatures at weight 34
calculating strong signatures at weight 36
calculating strong signatures at weight 38
calculating strong signatures at weight 40
calculating strong signatures at weight 42
calculating strong signatures at weight 44
calculating strong signatures at weight 46
calculating strong signatures at weight 48
calculating strong signatures at weight 50
calculating strong signatures at weight 52
calculating strong signatures at weight 54
calculating strong signatures at weight 56
calculating strong signatures at weight 58
calculating

In [10]:
signatures_by_weight_residue = {}

for k, a2, a19 in signatures:
    r = k % period

    if r not in signatures_by_weight_residue:
        signatures_by_weight_residue[r] = set()

    signatures_by_weight_residue[r].add((ZZ(a2), ZZ(a19)))

In [11]:
print(
    f"{'k mod ' + str(period):<10} | "
    f"(a_{l1}, a_{l2}) mod {p^m}"
)
print("-" * 70)

for r in sorted(signatures_by_weight_residue):
    pairs = ", ".join(
        f"({a1}, {a2})"
        for a1, a2 in sorted(signatures_by_weight_residue[r])
    )

    print(f"{str(r):<10} | {pairs}")

k mod 20   | (a_2, a_19) mod 25
----------------------------------------------------------------------
0          | (6, 15), (9, 20), (11, 5), (14, 5), (16, 20), (19, 15)
2          | (3, 20), (7, 5), (12, 10), (13, 10), (18, 5), (22, 20)
4          | (1, 15), (4, 0), (9, 10), (16, 10), (21, 0), (24, 15)
6          | (2, 10), (8, 0), (12, 20), (13, 20), (17, 0), (23, 10)
8          | (4, 15), (9, 0), (11, 10), (14, 10), (16, 0), (21, 15)
10         | (2, 20), (8, 10), (12, 5), (13, 5), (17, 10), (23, 20)
12         | (1, 20), (4, 5), (9, 15), (16, 15), (21, 5), (24, 20)
14         | (3, 0), (7, 10), (12, 15), (13, 15), (18, 10), (22, 0)
16         | (6, 0), (9, 5), (11, 15), (14, 15), (16, 5), (19, 0)
18         | (2, 15), (3, 10), (12, 0), (13, 0), (22, 10), (23, 15)


In [12]:
assert possible_signatures == signatures_by_weight_residue

print("EVERY POSSIBLE MODULO-25 SIGNATURE IS REALIZED")

EVERY POSSIBLE MODULO-25 SIGNATURE IS REALIZED
